# 02 — Embedding Analysis

**Goal:** Evaluate the quality of ResNet50 embeddings for product similarity search.

Covers:
- Embedding extraction from product images
- t-SNE / UMAP visualization (colored by category)
- Intra-class vs inter-class similarity distributions
- Qualitative search results (query → top-3 similar products)
- Category-level retrieval accuracy

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.manifold import TSNE
from collections import Counter

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. Load Embeddings

If embeddings exist on disk, load them. Otherwise, extract fresh from product images.

In [ ]:
EMBEDDING_PATH = Path('../models/embeddings.npy')
DATA_PATH = Path('../data/products.json')
INDEX_PATH = Path('../data/faiss_index')

# Load product metadata
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    products = json.load(f)
df = pd.DataFrame(products)

# Load or extract embeddings
if EMBEDDING_PATH.exists():
    embeddings = np.load(str(EMBEDDING_PATH))
    print(f'Loaded embeddings: {embeddings.shape}')
elif (INDEX_PATH / 'metadata.json').exists():
    # Load from FAISS index metadata
    import faiss
    index = faiss.read_index(str(INDEX_PATH / 'faiss_index.bin'))
    n = index.ntotal
    d = index.d
    embeddings = np.zeros((n, d), dtype='float32')
    for i in range(n):
        embeddings[i] = index.reconstruct(i)
    print(f'Reconstructed {n} embeddings from FAISS index')
    
    with open(INDEX_PATH / 'metadata.json') as f:
        metadata = json.load(f)
    categories = [metadata[str(i)].get('category', 'unknown') for i in range(n)]
else:
    print('No embeddings found. Run the index build first:')
    print('  curl -X POST http://localhost:8000/api/v1/index/build')
    embeddings = None

if embeddings is not None:
    print(f'Embedding shape: {embeddings.shape}')
    print(f'Embedding dtype: {embeddings.dtype}')
    norms = np.linalg.norm(embeddings, axis=1)
    print(f'L2 norms — mean: {norms.mean():.4f}, std: {norms.std():.6f}')

## 2. t-SNE Visualization

Project 2048-dim embeddings to 2D using t-SNE. Products from the same category should cluster together if the embeddings capture visual similarity well.

In [ ]:
if embeddings is not None:
    print('Running t-SNE (this may take ~30 seconds)...')
    tsne = TSNE(
        n_components=2,
        perplexity=30,
        random_state=42,
        n_iter=1000,
        learning_rate='auto',
        init='pca'
    )
    coords = tsne.fit_transform(embeddings)
    
    # Assign categories
    if 'categories' not in dir():
        categories = df['category'].tolist()[:len(embeddings)]
    
    unique_cats = sorted(set(categories))
    color_map = plt.cm.Set1(np.linspace(0, 1, len(unique_cats)))
    cat_to_color = {cat: color_map[i] for i, cat in enumerate(unique_cats)}
    
    fig, ax = plt.subplots(figsize=(12, 8))
    for cat in unique_cats:
        mask = [c == cat for c in categories]
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            label=cat, alpha=0.7, s=60,
            color=cat_to_color[cat], edgecolors='white', linewidth=0.5
        )
    
    ax.set_title('t-SNE of Product Embeddings (ResNet50)', fontsize=14, fontweight='bold')
    ax.set_xlabel('t-SNE Dimension 1')
    ax.set_ylabel('t-SNE Dimension 2')
    ax.legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('../data/tsne_embeddings.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: data/tsne_embeddings.png')

## 3. Similarity Distribution Analysis

Compare cosine similarity within categories (intra-class) vs between categories (inter-class). A good embedding space should show higher intra-class similarity.

In [ ]:
if embeddings is not None:
    # Compute full similarity matrix
    sim_matrix = embeddings @ embeddings.T
    
    intra_sims = []
    inter_sims = []
    
    n = len(categories)
    # Sample to keep computation manageable
    np.random.seed(42)
    sample_pairs = 5000
    
    for _ in range(sample_pairs):
        i, j = np.random.choice(n, 2, replace=False)
        sim = float(sim_matrix[i, j])
        if categories[i] == categories[j]:
            intra_sims.append(sim)
        else:
            inter_sims.append(sim)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(intra_sims, bins=50, alpha=0.6, label=f'Intra-class (n={len(intra_sims)})', color='green')
    ax.hist(inter_sims, bins=50, alpha=0.6, label=f'Inter-class (n={len(inter_sims)})', color='red')
    ax.axvline(np.mean(intra_sims), color='darkgreen', linestyle='--', label=f'Intra mean: {np.mean(intra_sims):.3f}')
    ax.axvline(np.mean(inter_sims), color='darkred', linestyle='--', label=f'Inter mean: {np.mean(inter_sims):.3f}')
    ax.set_title('Cosine Similarity Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Cosine Similarity')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../data/similarity_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Separation metric
    separation = np.mean(intra_sims) - np.mean(inter_sims)
    print(f'\nIntra-class similarity: {np.mean(intra_sims):.4f} ± {np.std(intra_sims):.4f}')
    print(f'Inter-class similarity: {np.mean(inter_sims):.4f} ± {np.std(inter_sims):.4f}')
    print(f'Separation gap: {separation:.4f}')
    print(f'\nInterpretation: {"GOOD" if separation > 0.05 else "WEAK"} category separation')

## 4. Per-Category Similarity Heatmap

In [ ]:
if embeddings is not None:
    unique_cats = sorted(set(categories))
    n_cats = len(unique_cats)
    
    # Mean similarity between each pair of categories
    cat_sim_matrix = np.zeros((n_cats, n_cats))
    for i, cat_i in enumerate(unique_cats):
        for j, cat_j in enumerate(unique_cats):
            mask_i = [c == cat_i for c in categories]
            mask_j = [c == cat_j for c in categories]
            emb_i = embeddings[mask_i]
            emb_j = embeddings[mask_j]
            sims = emb_i @ emb_j.T
            cat_sim_matrix[i, j] = sims.mean()
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cat_sim_matrix, cmap='YlOrRd', vmin=0.5, vmax=1.0)
    ax.set_xticks(range(n_cats))
    ax.set_yticks(range(n_cats))
    ax.set_xticklabels(unique_cats, rotation=45, ha='right')
    ax.set_yticklabels(unique_cats)
    
    # Add values
    for i in range(n_cats):
        for j in range(n_cats):
            ax.text(j, i, f'{cat_sim_matrix[i,j]:.2f}', ha='center', va='center', fontsize=9)
    
    plt.colorbar(im, label='Mean Cosine Similarity')
    ax.set_title('Inter-Category Similarity Heatmap', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('../data/category_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Retrieval Quality Evaluation

For each product, find top-3 similar products and measure how often they share the same category.

In [ ]:
if embeddings is not None:
    import faiss
    
    # Build temp index for evaluation
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    
    # Search top-4 for each product (first result is self)
    k = 4
    scores, indices = index.search(embeddings, k)
    
    # Calculate precision@3 (excluding self-match)
    correct = 0
    total = 0
    per_cat_precision = {cat: [] for cat in set(categories)}
    
    for i in range(len(embeddings)):
        query_cat = categories[i]
        neighbors = indices[i, 1:k]  # Skip index 0 (self)
        neighbor_cats = [categories[j] for j in neighbors if j >= 0]
        
        matches = sum(1 for c in neighbor_cats if c == query_cat)
        precision = matches / len(neighbor_cats) if neighbor_cats else 0
        per_cat_precision[query_cat].append(precision)
        correct += matches
        total += len(neighbor_cats)
    
    overall_precision = correct / total
    print(f'Overall Precision@3 (same category): {overall_precision:.1%}')
    print(f'\nPer-category Precision@3:')
    for cat in sorted(per_cat_precision.keys()):
        vals = per_cat_precision[cat]
        print(f'  {cat:15s}: {np.mean(vals):.1%} (n={len(vals)})')
    
    # Plot
    cats_sorted = sorted(per_cat_precision.keys())
    means = [np.mean(per_cat_precision[c]) for c in cats_sorted]
    
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(cats_sorted, means, color='steelblue')
    ax.axvline(overall_precision, color='red', linestyle='--', label=f'Overall: {overall_precision:.1%}')
    ax.set_xlabel('Precision@3 (Same Category)')
    ax.set_title('Category Retrieval Precision', fontsize=14, fontweight='bold')
    for bar, val in zip(bars, means):
        ax.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.1%}', va='center')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../data/retrieval_precision.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Summary

| Metric | Value |
|--------|-------|
| Embedding dimension | 2048 |
| Model | ResNet50 (ImageNet V2) |
| Normalization | L2 (verified: norms ≈ 1.0) |
| Category separation | Measurable via intra/inter similarity gap |
| Precision@3 | See results above |

**Key Insights:**
1. **t-SNE visualization** shows whether products from the same category cluster together visually
2. **Intra vs inter similarity** quantifies the embedding quality — larger gap = better features
3. **Precision@3** measures how useful the search system is in practice — what % of top-3 results share the same category as the query

**Improvement Paths:**
- Fine-tune ResNet50 on the jewellery domain (transfer learning) for better category separation
- Try DINOv2 or CLIP for potentially stronger visual features
- Add category-weighted search (boost same-category results)